In [11]:
import random

class VacuumEnvironment:
    def __init__(self):
        self.rooms = {
            "A": random.choice(["Clean", "Dirty"]),
            "B": random.choice(["Clean", "Dirty"])
        }

    def get_status(self, location):
        return self.rooms[location]

    def update(self, location, action):
        if action == "Suck":
            self.rooms[location] = "Clean"

class ReflexVacuumAgent:
    def act(self, location, status):
        if status == "Dirty":
            return "Suck"
        if location == "A":
            return "Right"
        return "Left"

env = VacuumEnvironment()
agent = ReflexVacuumAgent()
agent_location = random.choice(["A", "B"])
steps = 10

for i in range(steps):
    status = env.get_status(agent_location)
    action = agent.act(agent_location, status)
    env.update(agent_location, action)
    if action == "Right":
        agent_location = "B"
    elif action == "Left":
        agent_location = "A"
    print(f"Step {i} | Action: {action}")


Step 0 | Action: Right
Step 1 | Action: Left
Step 2 | Action: Right
Step 3 | Action: Left
Step 4 | Action: Right
Step 5 | Action: Left
Step 6 | Action: Right
Step 7 | Action: Left
Step 8 | Action: Right
Step 9 | Action: Left


## Experiment 2: Simple Reflex Agent in a 4-Room Grid Environment

In [12]:
class GridEnvironment:
    def __init__(self, size):
        self.size = size
        self.grid = {(x, y): "Dirty" for x in range(size) for y in range(size)}

    def get_status(self, position):
        return self.grid[position]

    def update(self, position, action):
        if action == "Suck":
            self.grid[position] = "Clean"

class ReflexAgent:
    def act(self, position, status, size):
        if status == "Dirty":
            return "Suck"
        moves = []
        x, y = position
        if x > 0: moves.append("Left")
        if x < size - 1: moves.append("Right")
        if y > 0: moves.append("Down")
        if y < size - 1: moves.append("Up")
        return random.choice(moves)

grid_size = 2
steps = 10

env = GridEnvironment(grid_size)
agent = ReflexAgent()
agent_pos = (0, 0)

for i in range(steps):
    status = env.get_status(agent_pos)
    action = agent.act(agent_pos, status, grid_size)
    print(f"Step {i} | Position: {agent_pos} | Status: {status} | Action: {action}")
    env.update(agent_pos, action)
    x, y = agent_pos
    if action == "Up":
        agent_pos = (x, y + 1)
    elif action == "Down":
        agent_pos = (x, y - 1)
    elif action == "Left":
        agent_pos = (x - 1, y)
    elif action == "Right":
        agent_pos = (x + 1, y)


Step 0 | Position: (0, 0) | Status: Dirty | Action: Suck
Step 1 | Position: (0, 0) | Status: Clean | Action: Up
Step 2 | Position: (0, 1) | Status: Dirty | Action: Suck
Step 3 | Position: (0, 1) | Status: Clean | Action: Down
Step 4 | Position: (0, 0) | Status: Clean | Action: Right
Step 5 | Position: (1, 0) | Status: Dirty | Action: Suck
Step 6 | Position: (1, 0) | Status: Clean | Action: Up
Step 7 | Position: (1, 1) | Status: Dirty | Action: Suck
Step 8 | Position: (1, 1) | Status: Clean | Action: Left
Step 9 | Position: (0, 1) | Status: Clean | Action: Right


## Experiment 3: Model-Based Vacuum Cleaner Agent

In [13]:
import random

class GridEnvironment:
    def __init__(self, size):
        self.size = size
        self.grid = {(x, y): "Dirty" for x in range(size) for y in range(size)}

    def get_status(self, position):
        return self.grid[position]

    def clean(self, position):
        self.grid[position] = "Clean"

class ModelBasedAgent:
    def __init__(self):
        self.model = {}

    def neighbors(self, position, size):
        x, y = position
        n = []
        if x > 0: n.append((x-1, y))
        if x < size - 1: n.append((x+1, y))
        if y > 0: n.append((x, y-1))
        if y < size - 1: n.append((x, y+1))
        return n

    def act(self, position, status, env):
        self.model[position] = status
        if status == "Dirty":
            self.model[position] = "Clean"
            return "Suck", position
        targets = [p for p in env.grid if self.model.get(p) != "Clean"]
        if not targets:
            return None, position
        options = [n for n in self.neighbors(position, env.size) if self.model.get(n) != "Clean"]
        if not options:
            options = self.neighbors(position, env.size)
        next_pos = random.choice(options)
        x, y = position
        nx, ny = next_pos
        if nx > x: action = "Right"
        elif nx < x: action = "Left"
        elif ny > y: action = "Up"
        else: action = "Down"
        return action, next_pos

grid_size = 2
steps = 10

env = GridEnvironment(grid_size)
agent = ModelBasedAgent()
agent_pos = (0, 0)

for i in range(steps):
    status = env.get_status(agent_pos)
    action, new_pos = agent.act(agent_pos, status, env)
    if action is None:
        print(f"Step {i} | All rooms clean")
        break
    print(f"Step {i} | Position: {agent_pos} | Action: {action}")
    if action == "Suck":
        env.clean(agent_pos)
    else:
        agent_pos = new_pos


Step 0 | Position: (0, 0) | Action: Suck
Step 1 | Position: (0, 0) | Action: Right
Step 2 | Position: (1, 0) | Action: Suck
Step 3 | Position: (1, 0) | Action: Up
Step 4 | Position: (1, 1) | Action: Suck
Step 5 | Position: (1, 1) | Action: Left
Step 6 | Position: (0, 1) | Action: Suck
Step 7 | All rooms clean


## 1. Why does the simple reflex agent perform redundant actions?

The simple reflex agent has no memory of past states or actions. Its decisions depend only on the current percept, so it cannot distinguish between rooms that have already been cleaned and those that have not. As a result, it may revisit clean rooms and perform unnecessary movements, leading to redundant actions.

## 2. How does the internal model improve agent performance?

The internal model allows the agent to store information about the cleanliness of rooms it has already visited. By using this stored knowledge, the agent can avoid revisiting rooms known to be clean and can prioritize moving toward dirty or unvisited rooms. This reduces unnecessary movements and improves overall efficiency.

## 3. Compare the simple reflex and model-based agents

### a. Rationality
The model-based agent is more rational because it uses both current percepts and past information to choose actions that better achieve the goal of cleaning all rooms efficiently.

### b. Efficiency
The model-based agent is more efficient since it minimizes redundant movements and completes the task using fewer actions compared to the simple reflex agent.

### c. Scalability
The model-based agent scales better in larger or more complex environments because its internal model helps manage increasing state complexity, whereas the simple reflex agent becomes increasingly inefficient as the environment grows.

## 4. Would the model-based agent work correctly in a partially observable environment? Justify.

In a partially observable environment, the model-based agent may not work correctly because its internal model relies on accurate and complete percepts. If the agent receives incomplete or incorrect information, the model can become inaccurate, leading to suboptimal or incorrect decisions unless additional mechanisms such as belief states or probabilistic reasoning are used.